In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import zipfile, os

zip_path = "/content/drive/MyDrive/agro-spray/betel.zip"
extract_path = "/content/data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done")

Done


In [7]:
for split in ['train', 'val', 'test']:
    for cls in ['healthy', 'diseased']:
        path = f"/content/data/betel/{split}/{cls}"
        count = len(os.listdir(path)) if os.path.exists(path) else "PATH NOT FOUND"
        print(f"{split}/{cls}: {count}")

train/healthy: 757
train/diseased: 669
val/healthy: 162
val/diseased: 143
test/healthy: 162
test/diseased: 143


In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU — go to Runtime → Change runtime type → T4 GPU")

GPU available: True
Device: Tesla T4


In [ ]:
import os
print(os.listdir("/content/data/betel"))

['val', 'train', 'test']


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])

In [ ]:
train_dataset = datasets.ImageFolder("/content/data/betel/train",transform = train_transforms)
val_dataset = datasets.ImageFolder("/content/data/betel/val",transform = val_transforms)

In [ ]:
print(train_dataset.classes)

['diseased', 'healthy']


In [ ]:
len(train_dataset),len(val_dataset)

(1426, 305)

In [ ]:
train_loader = DataLoader(train_dataset,batch_size = 32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle=False)

In [ ]:
model = models.resnet18(weights= models.ResNet18_Weights.IMAGENET1K_V1)

In [ ]:
#model

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr = 0.001)

In [ ]:
NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):

    # Training phase
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    # Validation phase
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_function(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train loss: {train_loss/train_total:.4f} "
          f"acc: {train_correct/train_total:.3f} | "
          f"Val loss: {val_loss/val_total:.4f} "
          f"acc: {val_correct/val_total:.3f}")

Epoch 1/10 | Train loss: 0.3062 acc: 0.879 | Val loss: 0.3360 acc: 0.856
Epoch 2/10 | Train loss: 0.2648 acc: 0.900 | Val loss: 0.3181 acc: 0.866
Epoch 3/10 | Train loss: 0.2461 acc: 0.903 | Val loss: 0.3040 acc: 0.869
Epoch 4/10 | Train loss: 0.2268 acc: 0.916 | Val loss: 0.3102 acc: 0.875
Epoch 5/10 | Train loss: 0.2369 acc: 0.911 | Val loss: 0.3022 acc: 0.872
Epoch 6/10 | Train loss: 0.2187 acc: 0.915 | Val loss: 0.2910 acc: 0.872
Epoch 7/10 | Train loss: 0.2250 acc: 0.908 | Val loss: 0.2880 acc: 0.885
Epoch 8/10 | Train loss: 0.2109 acc: 0.919 | Val loss: 0.2792 acc: 0.882
Epoch 9/10 | Train loss: 0.2277 acc: 0.904 | Val loss: 0.3240 acc: 0.866
Epoch 10/10 | Train loss: 0.2219 acc: 0.908 | Val loss: 0.2805 acc: 0.889


In [ ]:
torch.save({
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'val_acc': 0.889,
}, "/content/drive/MyDrive/agro-spray/betel_checkpoint.pth")